# LatentSpace: training pipeline on Kaggle (2x T4)

Tiny Aya → baseline → Aya Expanse teacher trajectories (rejection-filtered with the audit
rewards) → LoRA distillation → two GRPO runs (run_A: AgentTune's default tool reward,
run_B: audit-grounded reward) → held-out evaluation with pass^k and a bootstrap comparison.

**Before you start**
1. Settings → Accelerator: **GPU T4 x2**. Settings → Internet: **on** (needs a phone-verified account).
2. Add-ons → Secrets: add **`HF_TOKEN`** (a *write* token of the `kanishk001` account, with the
   Tiny Aya / Aya Expanse licences accepted). Optional: `GITHUB_TOKEN` if the repo is private.
3. Run the cells top to bottom. Each stage writes `results/NN_*.json`; download them at the end
   (or `PUSH = True` sends adapters, logs and datasets to the Hub).

T4 has no bf16: everything runs in fp16, training in 4-bit QLoRA, with gradient checkpointing.

## Time and cost estimate

| Stage | Where | Kaggle 2x T4 (wall) | GPU-h | HF credits |
|---|---|---|---|---|
| 0 install + tests | CPU | 8-12 min | - | - |
| 1 baseline, local fp16 | GPU 0 | ~25-35 min / 100 tasks | 0.5 | - |
| 1 baseline, router (optional) | API | ~5-10 min / 100 tasks (parallel to the local one) | - | ~$0.05-0.10 |
| 2 teacher_collect, aya-expanse-32b router | API | ~1.5-2 h / 600 trajectories (sequential calls) | - | ~$1.5-2.5, capped by `MAX_USD` |
| 2' teacher_collect, 8B 4-bit local fallback | GPU 0 | ~4-5 h / 300 trajectories | 4-5 | - |
| 3 distill, QLoRA 4-bit fp16 | GPU 0 | ~30-60 min / 400 rows x 2 epochs | 0.5-1 | - |
| 4 grpo run_A ‖ run_B, QLoRA, 100 steps | GPU 0 ‖ GPU 1 | ~3-3.5 h (both at once) | 6-7 | - |
| 5 evaluate, 4 checkpoints x 100 tasks x 5 trials, fp16 | GPU 0 ‖ GPU 1 | ~4-5 h | 8-10 | - |

Assumptions: Tiny Aya 3.35B; HF `generate` at ~20 tok/s (fp16) / ~10 tok/s (4-bit) per
sequence on a T4; ~300 generated tokens and ~2.5 turns per rollout; AgentTune rolls out one
trajectory at a time. Teacher cost at $0.50 / $1.50 per M input / output tokens (Cohere's list
price for Aya Expanse 32B; check the provider price on the model page). Scale linearly with the
number of tasks. Kaggle sessions stop at 12 h, so this notebook is meant to be run as three
sessions (stages 0-3, stage 4, stage 5); with `PUSH = True` each session picks up the previous
one's adapters and data from the Hub.

In [ ]:
# HF token from Kaggle Secrets (never printed)
import os
from kaggle_secrets import UserSecretsClient
secrets = UserSecretsClient()
os.environ["HF_TOKEN"] = secrets.get_secret("HF_TOKEN")
try:
    os.environ["GITHUB_TOKEN"] = secrets.get_secret("GITHUB_TOKEN")
except Exception:
    pass
print("HF_TOKEN set:", bool(os.environ.get("HF_TOKEN")))

In [ ]:
# Clone the repo and install AuditKIT + AgentTune (pinned) on top of Kaggle's CUDA torch
import os, subprocess
BRANCH = "training-pipeline"   # switch to "main" once merged
REPO = "/kaggle/working/LatentSpace"
gh = os.environ.get("GITHUB_TOKEN")
url = f"https://{gh}@github.com/kushwaha001/LatentSpace" if gh else "https://github.com/kushwaha001/LatentSpace"
if not os.path.isdir(REPO):
    subprocess.run(["git", "clone", "-q", "-b", BRANCH, url, REPO], check=True)
subprocess.run(["bash", f"{REPO}/scripts/install_deps.sh", "/kaggle/working/AgentTune"], check=True)

In [ ]:
import os, subprocess, sys, json, time, glob
os.chdir(REPO)
os.environ["PYTHONPATH"] = f"{REPO}/src"
os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")

def stage(*args, gpu=None, log=None):
    """Run one pipeline stage; stream its output. gpu=0/1 pins it to one GPU."""
    env = dict(os.environ)
    if gpu is not None:
        env["CUDA_VISIBLE_DEVICES"] = str(gpu)
    cmd = [sys.executable, "-m", "latentspace_train", *args]
    print("$", " ".join(cmd), f"(GPU {gpu})" if gpu is not None else "")
    if log:  # background: returns the process, output goes to the log file
        return subprocess.Popen(cmd, env=env, stdout=open(log, "w"), stderr=subprocess.STDOUT)
    p = subprocess.run(cmd, env=env)
    if p.returncode:
        raise RuntimeError(f"stage failed: {' '.join(args)}")

def wait_all(procs, logs, every=300):
    """Wait for background stages, printing the last log line every `every` seconds."""
    while any(p.poll() is None for p in procs):
        time.sleep(every)
        for l in logs:
            tail = open(l).read().strip().splitlines()[-1:] if os.path.exists(l) else []
            print(time.strftime("%H:%M"), l, "|", tail[0][-160:] if tail else "")
    for p, l in zip(procs, logs):
        print(l, "exit", p.returncode)
        if p.returncode:
            print(open(l).read()[-3000:])
    assert all(p.returncode == 0 for p in procs), "a background stage failed (log above)"

def show(path):
    d = json.load(open(path))
    print(path, "| seconds", d.get("seconds"), "| commit", d["provenance"]["commit"][:8])
    return d

In [ ]:
# ── Settings ────────────────────────────────────────────────────────────────
STUDENT     = "tiny-aya-global"   # or tiny-aya-fire / tiny-aya-water / tiny-aya-earth
PUSH        = True                # push adapters, logs, datasets to the Hub (needs a write token)
HF_USER     = "kanishk001"        # Hub namespace for every repo the pipeline creates
MAX_USD     = 5.0                 # teacher_collect budget (HF credits, Inference Providers)
API_BASELINE = True               # also run the cheap router baseline of tiny-aya-global
LIMIT       = None                # e.g. 20 for a quick end-to-end check on real data
TEACHER_BACKEND = "router"        # "router" (aya-expanse-32b, HF credits) or "local" (8B 4-bit on GPU)

COMMON = [f"--set=student.model={STUDENT}", f"--set=hub.namespace={HF_USER}"]
if LIMIT:
    COMMON.append(f"--limit={LIMIT}")
PUSH_FLAG = ["--push"] if PUSH else []
HUB = lambda name: f"{HF_USER}/{name}"

In [ ]:
# The transit tools (src/transit_agent/) and eval set (data/eval/) come from the eval-set
# branch. Merge it if this checkout does not have them yet; otherwise fall back to the stub.
if not (os.path.isdir(f"{REPO}/src/transit_agent") and os.path.isdir(f"{REPO}/data/eval")):
    r = subprocess.run(f"cd {REPO} && git fetch -q origin eval-set && "
                       "git -c user.name=nb -c user.email=nb@local merge -q --no-edit origin/eval-set",
                       shell=True)
    print("merged eval-set" if r.returncode == 0 else "eval-set branch not available")
if os.path.isdir(f"{REPO}/src/transit_agent") and os.path.isdir(f"{REPO}/data/eval"):
    print("transit tools + eval set present:", sorted(os.listdir(f"{REPO}/data/eval")))
else:
    print("WARNING: no transit tools / eval set -> using the stub tasks (wiring check only)")
    COMMON.append("--set=tasks.source=stub")

In [ ]:
# Sanity: GPUs, unit tests (CPU, ~15 s), and a CPU dry run of one GRPO preset (~30 s)
subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv"])
subprocess.run([sys.executable, "-m", "pytest", "-q", "tests"], check=True)
stage("grpo", "--run", "run_B", "--dry-run")

## Session 1: stages 1-3

In [ ]:
# Stage 1: baseline on the eval split. Local weights (fp16) on GPU 0; the optional router
# baseline needs no GPU and runs at the same time (its results: results/api_baseline/).
procs = [stage("baseline", *COMMON, *PUSH_FLAG, gpu=0, log="baseline_local.log")]
logs = ["baseline_local.log"]
if API_BASELINE:
    procs.append(stage("baseline", *COMMON, *PUSH_FLAG, "--set=baseline.modes=[api]",
                       "--set=results_dir=results/api_baseline", gpu="", log="baseline_api.log"))
    logs.append("baseline_api.log")
wait_all(procs, logs, every=120)
show("results/10_baseline.json")["modes"]["local"]["summary"]["overall"]

In [ ]:
# Stage 2: teacher trajectories (aya-expanse-32b via the HF router), stops at MAX_USD.
# TEACHER_BACKEND="local" uses aya-expanse-8b in 4-bit on GPU 0 instead (no credits).
stage("teacher_collect", *COMMON, *PUSH_FLAG, f"--max-usd={MAX_USD}",
      f"--set=teacher.backend={TEACHER_BACKEND}", gpu=0)
t = show("results/11_teacher_collect.json")
print("kept", t["kept"], "of", t["seen"], t["reasons"]); print(t["budget"])

In [ ]:
# Stage 3: LoRA distillation of the student on the kept trajectories (GPU 0)
stage("distill", *COMMON, *PUSH_FLAG, gpu=0)
d = show("results/12_distill.json"); print(d["n_rows"], "rows", d["train_result"])

## Session 2: stage 4, both GRPO runs at once (one per T4)

Identical seed, data, steps and hyperparameters; only the reward differs. If this is a new
session, re-run the setup cells above first; `RESUME_FROM_HUB` then loads the distilled
adapter from the Hub.

In [ ]:
RESUME_FROM_HUB = not os.path.isdir(f"{REPO}/outputs/distill/adapter")
START = [f"--set=student.adapter={HUB('tiny-aya-global-transit-distill-lora')}"] if RESUME_FROM_HUB else []
print("starting GRPO from", "Hub adapter" if RESUME_FROM_HUB else "outputs/distill/adapter")
procs = [stage("grpo", "--run", "run_A", *COMMON, *START, *PUSH_FLAG, gpu=0, log="grpo_run_A.log"),
         stage("grpo", "--run", "run_B", *COMMON, *START, *PUSH_FLAG, gpu=1, log="grpo_run_B.log")]
wait_all(procs, ["grpo_run_A.log", "grpo_run_B.log"])
for r in ("run_A", "run_B"):
    d = show(f"results/13_grpo_{r}.json")
    print(r, d["reward"]["kind"], "steps", d["global_step"], "error", d["error"],
          "fingerprint", d["shared_settings_fingerprint"])

## Session 3: stage 5, evaluation (two checkpoints per GPU), then the comparison

In [ ]:
def ck(name, repo):
    local = f"{REPO}/outputs/{'distill' if name == 'distill' else 'grpo/' + name}/adapter"
    return local if os.path.isdir(local) else HUB(repo)

ADAPTERS = {
    "distill": ck("distill", "tiny-aya-global-transit-distill-lora"),
    "run_A": ck("run_A", "tiny-aya-global-transit-grpo-default-reward-lora"),
    "run_B": ck("run_B", "tiny-aya-global-transit-grpo-audit-reward-lora"),
}
CKPTS = "[{name: base}, " + ", ".join(f"{{name: {n}, adapter: '{a}'}}" for n, a in ADAPTERS.items()) + "]"
EV = [*COMMON, *PUSH_FLAG, f"--set=evaluate.checkpoints={CKPTS}"]
procs = [stage("evaluate", *EV, "--run", "run_A", gpu=0, log="eval_run_A.log"),
         stage("evaluate", *EV, "--run", "run_B", gpu=1, log="eval_run_B.log")]
wait_all(procs, ["eval_run_A.log", "eval_run_B.log"])
procs = [stage("evaluate", *EV, "--run", "base", gpu=0, log="eval_base.log"),
         stage("evaluate", *EV, "--run", "distill", gpu=1, log="eval_distill.log")]
wait_all(procs, ["eval_base.log", "eval_distill.log"])
stage("evaluate", *EV, "--compare-only")   # summaries + AuditKIT paired bootstrap -> results/14_evaluate.json

In [ ]:
e = show("results/14_evaluate.json")
rows = []
for name, c in e["checkpoints"].items():
    if "summary" not in c:
        print(name, c); continue
    o = c["summary"]["overall"]; rel = o.get("reliability", {})
    rows.append((name, o["success"], rel.get("pass_hat_k"), o["tool_call_validity"], o["audit_combined"],
                 o["agent_loop_detection"], o["unconfirmed_irreversible_rate"]))
print(f"{'ckpt':8} {'success':>8} {'pass^5':>7} {'valid':>6} {'reward':>7} {'loopfree':>8} {'unconf':>6}")
for r in rows:
    print(f"{r[0]:8} " + " ".join(f"{x:>7.3f}" if isinstance(x, float) else f"{str(x):>7}" for x in r[1:]))
for k, v in e["comparisons"].items():
    s = v.get("metrics", {}).get("success", v)
    print(k, "success delta", s.get("delta"), "p", s.get("p_value"))
for name, c in e["checkpoints"].items():
    if "summary" in c:
        print(name, {l: m["success"] for l, m in c["summary"]["per_language"].items()})

In [ ]:
# Keep the results: they are small JSON files. Download results.zip from the Output tab and
# commit results/*.json to the repo (outputs/ stays local or on the Hub).
subprocess.run(f"cd {REPO} && zip -qr /kaggle/working/results.zip results", shell=True)
print(sorted(glob.glob(f"{REPO}/results/1*.json")))